In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("RetailLakehouse_BronzeToSilver") \
    .config("spark.jars.packages", 
            "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.4") \
    .config("spark.sql.extensions", 
            "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", 
            "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl",              
            "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version : {spark.version}")
print(f"Session active: {spark.sparkContext.appName}")

Spark version : 3.5.0
Session active: RetailLakehouse_BronzeToSilver


In [3]:
# ─────────────────────────────────────────
# READ RAW DIMENSION FILES FROM BRONZE
# ─────────────────────────────────────────

df_customers_raw = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("s3a://bronze/raw/customers/customer_master_full.csv")

df_products_raw = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("s3a://bronze/raw/products/product_catalog_full.csv")

df_stores_raw = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("s3a://bronze/raw/stores/store_locations.csv")

print(f"Customers raw : {df_customers_raw.count():,} rows")
print(f"Products raw  : {df_products_raw.count():,} rows")
print(f"Stores raw    : {df_stores_raw.count():,} rows")

# Verify schemas
df_customers_raw.printSchema()
df_products_raw.printSchema()
df_stores_raw.printSchema()

Customers raw : 10,000 rows
Products raw  : 1,000 rows
Stores raw    : 200 rows
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- zip_code: integer (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- created_date: date (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- list_price: double (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- is_active: boolean (nullable = true)

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- region: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- op

In [4]:
# ─────────────────────────────────────────
# READ RAW TRANSACTIONS FROM BRONZE
# ─────────────────────────────────────────

df_transactions_raw = spark.read \
    .option("inferSchema", True) \
    .parquet("s3a://bronze/raw/transactions/")

print(f"Transactions raw: {df_transactions_raw.count():,} rows")
print(f"Partitions      : {df_transactions_raw.rdd.getNumPartitions()}")
df_transactions_raw.printSchema()

Transactions raw: 500,000 rows
Partitions      : 12
root
 |-- transaction_id: string (nullable = true)
 |-- transaction_date: timestamp_ntz (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- order_line_num: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- unit_cost: double (nullable = true)
 |-- discount_amount: double (nullable = true)
 |-- net_revenue: double (nullable = true)
 |-- gross_profit: double (nullable = true)
 |-- tax_amount: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)



In [ ]:
from pyspark.sql.functions import (
    col, trim, upper, lower, to_date, 
    current_timestamp, lit, monotonically_increasing_id
)

# ─────────────────────────────────────────
# SILVER: CUSTOMERS
# ─────────────────────────────────────────
df_silver_customers = df_customers_raw \
    .withColumn("customer_name", trim(upper(col("customer_name")))) \
    .withColumn("email",         trim(lower(col("email")))) \
    .withColumn("created_date",  to_date(col("created_date"))) \
    .dropDuplicates(["customer_id"]) \
    .withColumn("_loaded_at", current_timestamp())

df_silver_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://silver/delta/customers")

print(f"Silver customers: {df_silver_customers.count():,} rows")

# ─────────────────────────────────────────
# SILVER: PRODUCTS
# ─────────────────────────────────────────
df_silver_products = df_products_raw \
    .withColumn("product_name", trim(col("product_name"))) \
    .withColumn("category",     trim(col("category"))) \
    .withColumn("subcategory",  trim(col("subcategory"))) \
    .withColumn("brand",        trim(col("brand"))) \
    .withColumn("list_price",   col("list_price").cast("decimal(10,2)")) \
    .withColumn("cost_price",   col("cost_price").cast("decimal(10,2)")) \
    .dropDuplicates(["product_id"]) \
    .withColumn("_loaded_at", current_timestamp())

df_silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://silver/delta/products")

print(f"Silver products : {df_silver_products.count():,} rows")

# ─────────────────────────────────────────
# SILVER: STORES
# ─────────────────────────────────────────
df_silver_stores = df_stores_raw \
    .withColumn("store_name",    trim(col("store_name"))) \
    .withColumn("city",          trim(col("city"))) \
    .withColumn("opening_date",  to_date(col("opening_date"))) \
    .dropDuplicates(["store_id"]) \
    .withColumn("_loaded_at", current_timestamp())

df_silver_stores.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://silver/delta/stores")

print(f"Silver stores   : {df_silver_stores.count():,} rows")

✔ Silver customers: 10,000 rows
✔ Silver products : 1,000 rows
✔ Silver stores   : 200 rows


In [ ]:
from pyspark.sql.functions import to_date, col, current_timestamp

# ─────────────────────────────────────────
# SILVER: TRANSACTIONS
# ─────────────────────────────────────────
df_silver_transactions = df_transactions_raw \
    .withColumn("transaction_date", to_date(col("transaction_date"))) \
    .filter(col("quantity") > 0) \
    .filter(col("unit_price") > 0) \
    .filter(col("net_revenue").isNotNull()) \
    .dropDuplicates(["transaction_id"]) \
    .withColumn("_loaded_at", current_timestamp())

df_silver_transactions.write \
    .format("delta") \
    .mode("overwrite") \
    .save("s3a://silver/delta/transactions")

print(f"Silver transactions: {df_silver_transactions.count():,} rows")

✔ Silver transactions: 500,000 rows


In [ ]:
# ─────────────────────────────────────────
# VERIFY SILVER DELTA TABLES
# ─────────────────────────────────────────
tables = ["customers", "products", "stores", "transactions"]

for table in tables:
    df = spark.read.format("delta").load(f"s3a://silver/delta/{table}")
    print(f"silver/delta/{table}: {df.count():,} rows | "
          f"{len(df.columns)} columns")

✔ silver/delta/customers: 10,000 rows | 9 columns
✔ silver/delta/products: 1,000 rows | 9 columns
✔ silver/delta/stores: 200 rows | 8 columns
✔ silver/delta/transactions: 500,000 rows | 17 columns
